In [1]:
import pandas as pd

# Jobs and Tech

In [2]:
jobs_agg_tech = pd.read_csv('transformed data/1-Jobs Agg Tech Skills.csv')
jobs_agg_tech

,O*NET-SOC Code,Job Title,TOTAL_TECH_SKILLS,TOTAL_TECH_TYPES,TECH_SKILLS_LIST,TECH_TYPES_LIST,PCT_HOT_TECH,NUM_HOT_TECH,PCT_IN_DEMAND_TECH,NUM_IN_DEMAND_TECH
0,15-1252.00,Software Developers,429,67,"""3M Post-it App""; ""A programming language APL""...","""Access software""; ""Administration software""; ...",34.498834,148,7.692308,33
1,15-1253.00,Software Quality Assurance Analysts and Testers,428,68,"""3M Post-it App""; ""A programming language APL""...","""Access software""; ""Accounting software""; ""Adm...",28.971963,124,5.140187,22
2,15-1299.09,Information Technology Project Managers,334,62,"""24SevenOffice Project""; ""3M Post-it App""; ""AE...","""Access software""; ""Accounting software""; ""Ana...",36.227545,121,2.694611,9
3,15-1243.00,Database Architects,324,59,"""3M Post-it App""; ""ADO.NET""; ""AJAX""; ""ASG Tech...","""Access software""; ""Administration software""; ...",34.567901,112,4.012346,13
4,15-1211.00,Computer Systems Analysts,313,68,"""3M Post-it App""; ""ADP Workforce Now""; ""AJAX"";...","""Access software""; ""Accounting software""; ""Adm...",38.019169,119,3.514377,11
...,...,...,...,...,...,...,...,...,...,...
918,35-9021.00,Dishwashers,2,2,"""Facebook""; ""Microsoft Windows""","""Operating system software""; ""Web page creatio...",100.000000,2,0.000000,0
919,53-7041.00,Hoist and Winch Operators,2,2,"""Microsoft Excel""; ""Microsoft Word""","""Spreadsheet software""; ""Word processing softw...",100.000000,2,0.000000,0
920,53-4041.00,Subway and Streetcar Operators,2,2,"""Microsoft Office software""; ""Word processing ...","""Office suite software""; ""Word processing soft...",50.000000,1,0.000000,0
921,51-4023.00,"Rolling Machine Setters, Operators, and Tender...",2,2,"""Email software""; ""Web browser software""","""Electronic mail software""; ""Internet browser ...",0.000000,0,0.000000,0


In [15]:
tech_agg_jobs = pd.read_csv('transformed data/1-Tech Skills Agg Jobs.csv')
tech_agg_jobs['TECH_SKILL_UNIQUENESS_RANK'] = tech_agg_jobs['TOTAL_JOBS_PER_TECH_SKILL'].rank(ascending=True)
tech_agg_jobs['UNIQUENESS_INV'] = 1 / tech_agg_jobs['TOTAL_JOBS_PER_TECH_SKILL']
tech_agg_jobs

,Tech Skill Title,Tech Type Title,Tech Type Code,Hot Technology,In Demand,TOTAL_JOBS_PER_TECH_SKILL,TOTAL_JOBS_PER_TECH_TYPE,JOB_TITLE_LIST,TECH_SKILL_UNIQUENESS_RANK,UNIQUENESS_INV
0,Adobe Acrobat,Document management software,43232202,1,0,165,293,"""Accountants and Auditors""; ""Administrative La...",8770.0,0.006061
1,AdSense Tracker,Data base user interface and query software,43232306,0,0,2,632,"""Chief Executives""; ""Marketing Managers""",6839.5,0.500000
2,Atlassian JIRA,Content workflow software,43232201,1,0,53,46,"""Administrative Services Managers""; ""Aerospace...",8706.5,0.018868
3,Blackbaud The Raiser's Edge,Customer relationship management CRM software,43232303,0,0,44,155,"""Accountants and Auditors""; ""Administrative Se...",8677.5,0.022727
4,ComputerEase construction accounting software,Accounting software,43231601,0,0,3,178,"""Bookkeeping, Accounting, and Auditing Clerks""...",7715.0,0.333333
...,...,...,...,...,...,...,...,...,...,...
8780,AMCS Platform,Analytical or scientific software,43232605,0,0,1,372,"""Refuse and Recyclable Material Collectors""",3073.0,1.000000
8781,Dossier software,Data base user interface and query software,43232306,0,0,1,632,"""Refuse and Recyclable Material Collectors""",3073.0,1.000000
8782,Mileage logging software,Data base user interface and query software,43232306,0,0,1,632,"""Refuse and Recyclable Material Collectors""",3073.0,1.000000
8783,Routeware software,Map creation software,43233506,0,0,1,87,"""Refuse and Recyclable Material Collectors""",3073.0,1.000000


In [19]:
tech_agg_jobs.columns

Index(['Tech Skill Title', 'Tech Type Title', 'Tech Type Code',
       'Hot Technology', 'In Demand', 'TOTAL_JOBS_PER_TECH_SKILL',
       'TOTAL_JOBS_PER_TECH_TYPE', 'JOB_TITLE_LIST',
       'TECH_SKILL_UNIQUENESS_RANK', 'UNIQUENESS_INV'],
      dtype='object')

# Average uniqueness

In [20]:
import pandas as pd

df = tech_agg_jobs.copy()

# 1) Turn the JOB_TITLE_LIST string into a real Python list
#    Example input:  '"Accountants and Auditors"; "Administrative ..."; ...'
df["JOB_TITLE_LIST_PARSED"] = (
    df["JOB_TITLE_LIST"]
      .fillna("")
      .astype(str)
      .str.replace('"', '', regex=False)   # remove quotes
      .str.split(r"\s*;\s*", regex=True)   # split on semicolons with optional spaces
)

# 2) Explode so each row is one (job title, tech skill) pair
job_skill = df.explode("JOB_TITLE_LIST_PARSED").rename(
    columns={"JOB_TITLE_LIST_PARSED": "Job Title"}
)

# 3) Clean any blanks that come from empty strings / trailing separators
job_skill["Job Title"] = job_skill["Job Title"].str.strip()
job_skill = job_skill[job_skill["Job Title"].ne("")]

# 4) Average UNIQUENESS_INV across all tech skills used by each job
job_uniqueness = (
    job_skill.groupby("Job Title", as_index=False)
             .agg(
                 AVG_TECH_SKILL_UNIQUENESS_INV=("UNIQUENESS_INV", "mean"),
                 NUM_TECH_SKILLS=("UNIQUENESS_INV", "size")
             )
             .sort_values("AVG_TECH_SKILL_UNIQUENESS_INV", ascending=False)
)

job_uniqueness

,Job Title,AVG_TECH_SKILL_UNIQUENESS_INV,NUM_TECH_SKILLS
582,Musical Instrument Repairers and Tuners,1.000000,10
761,"Roof Bolters, Mining",1.000000,2
123,Cement Masons and Concrete Finishers,0.933333,10
580,Music Directors and Composers,0.886508,86
151,"Coil Winders, Tapers, and Finishers",0.833333,3
...,...,...,...
212,"Cutters and Trimmers, Hand",0.001332,3
428,"Helpers--Pipelayers, Plumbers, Pipefitters, an...",0.001221,3
436,Hoist and Winch Operators,0.001220,2
408,"Grinding and Polishing Workers, Hand",0.001220,2


In [ ]:
import numpy as np

df = tech_agg_jobs.copy()

df["UNIQUENESS_INV_WITHIN_TYPE"] = df["TOTAL_JOBS_PER_TECH_TYPE"] / df["TOTAL_JOBS_PER_TECH_SKILL"]

# Then, find average tech skill uniqueness for each job, also weighing it with the overall number of tech skills used.